# Stage A2 — Google Colab Runtime Qualification & Hardware Discovery Notebook (Protocol V1.5)

**Protocol Specification:** Stage A2 V1.5 (Protocol Amendment 12)
**Dataset:** HDFS (`SPL-HDFS-001` Canonical Split Authority)
**Scope:** Pre-execution Environment Discovery, Target Framework Verification & CUDA Deterministic Qualification
**Hardware Policy:** `DYNAMIC_DISCOVER_THEN_LOCK` (Supports T4, L4, A100)
**Target Framework:** PyTorch `2.6.0+cu124` | CUDA `12.4` | `CUBLAS_WORKSPACE_CONFIG=:4096:8`

### Runtime Notes & Container Compatibility
- **Preferred Starting Container:** Colab Runtime Version `2025.07` (Python 3.11.x, PyTorch 2.6.0 starting image).
- **Machine-Verified Contract:** All runtime properties (PyTorch build, CUDA runtime, GPU device, driver) are verified fail-closed via Python subprocesses. No silent fallback to default Colab versions is permitted.
- **Zero Real Optimizer Steps:** Real empirical optimizer training is strictly gated and NOT authorized within this notebook.

In [ ]:
# CELL 1 — Mount Google Drive Durable Storage
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 2 — Runtime & GPU Discovery
!nvidia-smi
import sys, platform, torch
print('Python Version:', platform.python_version())
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name:', torch.cuda.get_device_name(0))
    print('CUDA Runtime:', torch.version.cuda)
    props = torch.cuda.get_device_properties(0)
    print(f'Compute Capability: {props.major}.{props.minor}')
    print(f'Total VRAM: {props.total_memory / (1024**3):.2f} GB')


In [ ]:
# CELL 3 — Clone Repository into /content/Research & Checkout Approved Frozen Commit
import os, sys, subprocess

# ============================================================================
# RUNTIME PARAMETER: Set the approved commit SHA supplied after review
# ============================================================================
APPROVED_PREPARATION_COMMIT = "<supplied-after-independent-review>"

if not APPROVED_PREPARATION_COMMIT or "<" in APPROVED_PREPARATION_COMMIT or len(APPROVED_PREPARATION_COMMIT.strip()) != 40:
    raise RuntimeError(
        f"FATAL: APPROVED_PREPARATION_COMMIT must be set to a valid 40-character hex commit SHA before execution! "
        f"Current value: {APPROVED_PREPARATION_COMMIT!r}"
    )

APPROVED_PREPARATION_COMMIT = APPROVED_PREPARATION_COMMIT.strip()

%cd /content
!rm -rf /content/Research
!git clone https://github.com/Minhlike/Chuyende.git /content/Research
%cd /content/Research
!git checkout {APPROVED_PREPARATION_COMMIT}

# Assert detached HEAD equals APPROVED_PREPARATION_COMMIT exactly
actual_head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd='/content/Research', text=True).strip()
print(f'Verified HEAD: {actual_head}')
assert actual_head == APPROVED_PREPARATION_COMMIT, f'FATAL: Checked out HEAD ({actual_head}) != APPROVED_PREPARATION_COMMIT ({APPROVED_PREPARATION_COMMIT})'

# Verify clean execution source tree
dirty_status = subprocess.check_output([
    'git', 'status', '--porcelain', 'src', 'tests', 'scripts', 'experiments'
], cwd='/content/Research', text=True).strip()
if dirty_status:
    raise RuntimeError(f'FATAL: Execution source tree has uncommitted modifications:\n{dirty_status}')

print(f'Frozen Source Verification: 100% PASS (Detached HEAD at {actual_head})')


In [ ]:
# CELL 4 — Dependency & Exact PyTorch 2.6.0+cu124 Framework Verification
import sys, subprocess

# 1. Install local research package from pyproject.toml (Fail-Closed, Editable Mode)
%cd /content/Research
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

# 2. Inspect current PyTorch build in a fresh subprocess
check_script = (
    'import sys, torch\n'
    'is_exact = (torch.__version__.startswith("2.6.0") and (torch.version.cuda == "12.4" or "cu124" in torch.__version__) and torch.cuda.is_available())\n'
    'print(f"TORCH_VER={torch.__version__},CUDA_VER={torch.version.cuda},CUDA_AVAIL={torch.cuda.is_available()},EXACT={is_exact}")\n'
    'sys.exit(0 if is_exact else 1)\n'
)
res = subprocess.run([sys.executable, '-c', check_script], text=True, capture_output=True)
print(res.stdout.strip())

# 3. If not exact, install official PyTorch 2.6.0 CUDA 12.4 wheel
if res.returncode != 0:
    print('Installing official PyTorch 2.6.0+cu124 wheel...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
        'torch==2.6.0',
        '--index-url', 'https://download.pytorch.org/whl/cu124'
    ], check=True)

# 4. Final verification via fresh subprocess
verify_script = (
    'import sys, torch\n'
    'print("Verified PyTorch:", torch.__version__, "| CUDA Runtime:", torch.version.cuda, "| CUDA Available:", torch.cuda.is_available())\n'
    'assert torch.__version__.startswith("2.6.0"), f"PyTorch version mismatch: {torch.__version__}"\n'
    'assert torch.version.cuda == "12.4", f"CUDA runtime mismatch: {torch.version.cuda}"\n'
    'assert torch.cuda.is_available(), "CUDA is not available"\n'
)
subprocess.run([sys.executable, '-c', verify_script], check=True)
print('PyTorch Framework Verification: EXACT 2.6.0+cu124 (CUDA 12.4) PASS.')


In [ ]:
# CELL 5 — Locate Drive HDFS Tarball, Copy to Fast Local Disk, Streaming SHA-256
import os, shutil, hashlib
from pathlib import Path

def compute_sha256_streaming(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    hasher = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()

drive_source = Path('/content/drive/MyDrive/Chuyende-stage-a2/datasets/HDFS_1.tar.gz')
if not drive_source.exists():
    drive_source = Path('/content/drive/MyDrive/HDFS_1.tar.gz')

local_dest = Path('/content/stage-a2-data/HDFS_1.tar.gz')
local_dest.parent.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA = '6ca6c5bc2671c66afecee9369a2fdac606bf33997a2494ac66aa411fe3e95169'

if drive_source.exists():
    print('Source on Drive found:', drive_source)
    src_sha = compute_sha256_streaming(drive_source)
    print('Source SHA-256:', src_sha)
    assert src_sha == EXPECTED_SHA, f'Source SHA mismatch: {src_sha} != {EXPECTED_SHA}'
    print('Copying to fast local disk:', local_dest)
    shutil.copy2(drive_source, local_dest)

assert local_dest.exists(), f'Local dataset missing at {local_dest}'
dst_sha = compute_sha256_streaming(local_dest)
print('Local Copy SHA-256:', dst_sha)
assert dst_sha == EXPECTED_SHA, f'Local copy SHA mismatch: {dst_sha} != {EXPECTED_SHA}'
print('HDFS Data Parity Verified: 100% MATCH.')


In [ ]:
# CELL 6 — Run Colab Bootstrap & Dynamic Hardware Discovery
%cd /content/Research
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
!python scripts/bootstrap_stage_a2_colab.py \
    --repo-dir /content/Research \
    --local-data-dest /content/stage-a2-data/HDFS_1.tar.gz \
    --durable-root /content/drive/MyDrive/Chuyende-stage-a2 \
    --env-lock-output /content/Research/experiments/evidence/stage-a2/preexecution/STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json


In [ ]:
# CELL 7 — Run NON_EMPIRICAL CUDA Deterministic Qualification
%cd /content/Research
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
!python scripts/run_stage_a2_deterministic_qualification.py \
    --device cuda \
    --base-dir /content/Research \
    --environment-lock /content/Research/experiments/evidence/stage-a2/preexecution/STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json


In [ ]:
# CELL 8 — Run Seed-42 Dry-Run Only (Zero Optimizer Steps)
%cd /content/Research
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
!python scripts/run_stage_a2_five_seed_empirical.py \
    --seed 42 \
    --dry-run \
    --base-dir /content/Research \
    --dataset-path /content/stage-a2-data/HDFS_1.tar.gz \
    --durable-root /content/drive/MyDrive/Chuyende-stage-a2/runs \
    --plan /content/Research/experiments/plans/STAGE-A2-FIVE-SEED-EXECUTION-PLAN-V1.5.json \
    --environment-lock /content/Research/experiments/evidence/stage-a2/preexecution/STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json


In [ ]:
# CELL 9 — Durably Mirror Qualification Artifacts to Google Drive
%cd /content/Research
!python scripts/bootstrap_stage_a2_colab.py \
    --mirror-qualification \
    --repo-dir /content/Research \
    --durable-root /content/drive/MyDrive/Chuyende-stage-a2


In [ ]:
# CELL 10 — STOP FOR INDEPENDENT AUTHORIZATION
# ============================================================================
# REAL EMPIRICAL TRAINING IS NOT AUTHORIZED IN THIS PREPARATION NOTEBOOK.
# Copy the durable qualification directory from Google Drive for independent audit.
# ============================================================================
print('=================================================================')
print('STAGE A2 COLAB PREPARATION & QUALIFICATION COMPLETE.')
print('Durable Qualification Artifacts Stored Under:')
print('/content/drive/MyDrive/Chuyende-stage-a2/qualification/')
print('STATUS: PENDING INDEPENDENT LAUNCH AUTHORIZATION FOR SEED 42.')
print('=================================================================')
